# polarctic Feature Tour

This notebook shows how polarctic exposes ArcticDB symbols as Polars LazyFrames.

It covers:
- the URI, Library, and LazyDataFrame entry points
- schema discovery
- filter, projection, and row-limit pushdown
- supported predicate translations such as `is_in`, `str.contains`, arithmetic, and `is_null`
- versioned reads with `as_of`

The examples assume the notebook is running from the repository's Python environment, for example after `pip install -e ".[dev]"`.

In [ ]:
import gc
from tempfile import TemporaryDirectory

import pandas as pd
import polars as pl
from arcticdb import Arctic, OutputFormat, QueryBuilder

from polarctic import scan_arcticdb

In [ ]:
tmp_dir = TemporaryDirectory(
    prefix="polarctic_feature_tour_"
)
uri = f"lmdb://{tmp_dir.name}"
lib_name = "feature_demo"
symbol = "trading_book"

writer = Arctic(uri)
writer_lib = writer.create_library(lib_name)

trading_book = pd.DataFrame(
    {
        "desk": ["rates", "rates", "fx", "fx", "credit", "commodities", "equities", "rates"],
        "region": ["LDN", "NYC", "LDN", "SGP", "NYC", None, "LDN", "SGP"],
        "trades": [120, 75, 90, 40, 55, 35, 68, 88],
        "pnl": [1.8, -0.4, 0.7, 1.1, -0.1, 0.3, 0.5, -0.2],
        "risk": [5, 3, 4, 2, 6, 1, 2, 4],
        "asof": pd.date_range("2025-01-02", periods=8, freq="D"),
    }
)

writer_lib.write(symbol, trading_book)
del writer_lib
del writer
gc.collect()

trading_book

,desk,region,trades,pnl,risk,asof
0,rates,LDN,120,1.8,5,2025-01-02
1,rates,NYC,75,-0.4,3,2025-01-03
2,fx,LDN,90,0.7,4,2025-01-04
3,fx,SGP,40,1.1,2,2025-01-05
4,credit,NYC,55,-0.1,6,2025-01-06
5,commodities,None,35,0.3,1,2025-01-07
6,equities,LDN,68,0.5,2,2025-01-08
7,rates,SGP,88,-0.2,4,2025-01-09


## 1. Read the same symbol through the URI and Library forms

`scan_arcticdb()` accepts either a URI plus library name, or an already-open ArcticDB library. Both return the same Polars `LazyFrame`.

In [11]:
uri_result = scan_arcticdb(uri, lib_name, symbol).collect()
gc.collect()

ac = Arctic(uri)
lib = ac.get_library(lib_name)
library_result = scan_arcticdb(lib, symbol).collect()

print(f"Library form matches URI form: {library_result.equals(uri_result)}")
uri_result

Library form matches URI form: True


desk,region,trades,pnl,risk,asof
str,str,i64,f64,i64,datetime[ns]
"""rates""","""LDN""",120,1.8,5,2025-01-02 00:00:00
"""rates""","""NYC""",75,-0.4,3,2025-01-03 00:00:00
"""fx""","""LDN""",90,0.7,4,2025-01-04 00:00:00
"""fx""","""SGP""",40,1.1,2,2025-01-05 00:00:00
"""credit""","""NYC""",55,-0.1,6,2025-01-06 00:00:00
"""commodities""",null,35,0.3,1,2025-01-07 00:00:00
"""equities""","""LDN""",68,0.5,2,2025-01-08 00:00:00
"""rates""","""SGP""",88,-0.2,4,2025-01-09 00:00:00


In [12]:
scan_arcticdb(lib, symbol).collect_schema()

Schema([('desk', String),
        ('region', String),
        ('trades', Int64),
        ('pnl', Float64),
        ('risk', Int64),
        ('asof', Datetime(time_unit='ns', time_zone=None))])

## 2. Use Polars filters, projection, and row limits

polarctic pushes supported filters into ArcticDB, applies column projection when you `select`, and respects Polars row limits such as `head`.

In [13]:
pushdown_result = (
    scan_arcticdb(lib, symbol)
    .filter((pl.col("trades") >= 60) & pl.col("desk").is_in(["rates", "fx"]))
    .select("desk", "region", "trades", "pnl")
    .head(3)
    .collect()
)

pushdown_result

desk,region,trades,pnl
str,str,i64,f64
"""rates""","""LDN""",120,1.8
"""rates""","""NYC""",75,-0.4
"""fx""","""LDN""",90,0.7


## 3. Exercise some translated predicate features

The translator supports comparisons, arithmetic inside predicates, boolean composition, `is_in`, `str.contains`, `is_null`, `abs`, and unary negation. This example uses several of those in one query.

In [14]:
predicate_demo = (
    scan_arcticdb(lib, symbol)
    .filter(
        (((pl.col("trades") + pl.col("risk")) > 80) & pl.col("desk").str.contains("rates|fx"))
        | pl.col("region").is_null()
    )
    .select("desk", "region", "trades", "risk", "pnl")
    .collect()
)

predicate_demo

desk,region,trades,risk,pnl
str,str,i64,i64,f64
"""rates""","""LDN""",120,5,1.8
"""fx""","""LDN""",90,4,0.7
"""commodities""",null,35,1,0.3
"""rates""","""SGP""",88,4,-0.2


## 4. Start from an ArcticDB `LazyDataFrame`

You can hand `scan_arcticdb()` an ArcticDB `LazyDataFrame` that already has Arctic-side processing attached. Additional Polars predicates and projections then stack on top of that base read.

In [15]:
qb = QueryBuilder()
qb = qb[qb["trades"] >= 50]

lazy_source = lib.read(
    symbol,
    query_builder=qb,
    lazy=True,
    output_format=OutputFormat.PYARROW,
)
lazy_source["load"] = lazy_source["trades"] * lazy_source["risk"]

lazy_source_result = (
    scan_arcticdb(lazy_source)
    .filter(pl.col("pnl") > 0)
    .select("desk", "trades", "risk", "load", "pnl")
    .collect()
)

lazy_source_result

desk,trades,risk,load,pnl
str,i64,i64,i64,f64
"""rates""",120,5,600,1.8
"""fx""",90,4,360,0.7
"""equities""",68,2,136,0.5


## 5. Read older versions with `as_of`

Because `scan_arcticdb()` passes `as_of` through to ArcticDB, you can expose previous symbol versions through the same Polars API.

In [16]:
updated_book = trading_book.copy()
updated_book.loc[updated_book["desk"].eq("rates"), "pnl"] += 0.5
lib.write(symbol, updated_book)

original_view = (
    scan_arcticdb(lib, symbol, as_of=0)
    .select("desk", "pnl")
    .with_columns(pl.lit("as_of=0").alias("version"))
    .collect()
)

latest_view = (
    scan_arcticdb(lib, symbol)
    .select("desk", "pnl")
    .with_columns(pl.lit("latest").alias("version"))
    .collect()
)

pl.concat([original_view, latest_view]).select("version", "desk", "pnl").sort(["desk", "version"])

version,desk,pnl
str,str,f64
"""as_of=0""","""commodities""",0.3
"""latest""","""commodities""",0.3
"""as_of=0""","""credit""",-0.1
"""latest""","""credit""",-0.1
"""as_of=0""","""equities""",0.5
…,…,…
"""as_of=0""","""rates""",-0.4
"""as_of=0""","""rates""",-0.2
"""latest""","""rates""",2.3


The temporary LMDB directory is kept alive by `tmp_dir` while the kernel is running and will be cleaned up automatically when the notebook session ends.